# Phase 1: Data Loading & Inspection

### Step 1: Data Loading

In [277]:
import pandas as pd
import numpy as np
from IPython.display import display
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import MinMaxScaler


# 1. Load the data 
# Using "../" goes up one directory level to find the "data" folder
file_path = "../data/risk_factors_cervical_cancer.csv"

df = pd.read_csv(file_path)

# 2. Check the shape
rows, cols = df.shape
print(f"Dataset Shape: {rows} patients and {cols} features\n")

# Peek at the first few rows
display(df.head())

Dataset Shape: 858 patients and 36 features



,Age,Number of sexual partners,First sexual intercourse,Num of pregnancies,Smokes,Smokes (years),Smokes (packs/year),Hormonal Contraceptives,Hormonal Contraceptives (years),IUD,...,STDs: Time since first diagnosis,STDs: Time since last diagnosis,Dx:Cancer,Dx:CIN,Dx:HPV,Dx,Hinselmann,Schiller,Citology,Biopsy
0,18,4.0,15.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
1,15,1.0,14.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
2,34,1.0,?,1.0,0.0,0.0,0.0,0.0,0.0,0.0,...,?,?,0,0,0,0,0,0,0,0
3,52,5.0,16.0,4.0,1.0,37.0,37.0,1.0,3.0,0.0,...,?,?,1,0,1,0,0,0,0,0
4,46,3.0,21.0,4.0,0.0,0.0,0.0,1.0,15.0,0.0,...,?,?,0,0,0,0,0,0,0,0


### Step 2: Initial Diagnostic Checks

In [278]:
# 3. Inspect Data Types
print("--- DataFrame Info ---")
df.info()
print("\n")

# 4. Investigate Suspicious 'Object' Columns
suspicious_col = 'Number of sexual partners' 
print(f"--- Unique values in '{suspicious_col}' ---")
print(df[suspicious_col].unique()) 
print("\n")

# 5. Identify the Target Variable
target = 'Biopsy'
print(f"--- Target Variable Distribution ({target}) ---")
display(df[target].value_counts(normalize=True).round(4) * 100)

--- DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 858 entries, 0 to 857
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype 
---  ------                              --------------  ----- 
 0   Age                                 858 non-null    int64 
 1   Number of sexual partners           858 non-null    object
 2   First sexual intercourse            858 non-null    object
 3   Num of pregnancies                  858 non-null    object
 4   Smokes                              858 non-null    object
 5   Smokes (years)                      858 non-null    object
 6   Smokes (packs/year)                 858 non-null    object
 7   Hormonal Contraceptives             858 non-null    object
 8   Hormonal Contraceptives (years)     858 non-null    object
 9   IUD                                 858 non-null    object
 10  IUD (years)                         858 non-null    object
 11  STDs                               

Biopsy
0    93.59
1     6.41
Name: proportion, dtype: float64

# Phase 2: Data Cleaning & Type Conversion

In [279]:
# 1. Replace the text '?' with standard numpy NaN
df = df.replace('?', np.nan)

## 2. Convert to numeric, explicitly flagging failures
failed_cols = []
for col in df.columns:
    try:
        df[col] = pd.to_numeric(df[col])
    except ValueError:
        failed_cols.append(col)

print("Data types conversion attempted.\n")
if failed_cols:
    print(f"⚠️ WARNING: The following columns failed conversion and remain as text/objects:")
    print(failed_cols)
    print("Investigate these manually before proceeding!\n")
else:
    print("All columns successfully converted to numeric!\n")

print("--- Updated DataFrame Info ---")
df.info()

Data types conversion attempted.

All columns successfully converted to numeric!

--- Updated DataFrame Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 858 entries, 0 to 857
Data columns (total 36 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   Age                                 858 non-null    int64  
 1   Number of sexual partners           832 non-null    float64
 2   First sexual intercourse            851 non-null    float64
 3   Num of pregnancies                  802 non-null    float64
 4   Smokes                              845 non-null    float64
 5   Smokes (years)                      845 non-null    float64
 6   Smokes (packs/year)                 845 non-null    float64
 7   Hormonal Contraceptives             750 non-null    float64
 8   Hormonal Contraceptives (years)     750 non-null    float64
 9   IUD                                 741 non-null    float64
 1

# Phase 3: Data Assessment & Investigation

### Step 1: Consistency check

In [280]:
import pandas as pd
import numpy as np

print("======================================================")
print("PHASE 3: CLINICAL & LOGICAL DATA ASSESSMENT PIPELINE")
print("======================================================")
print("Note: Row IDs refer to the pandas DataFrame Index.")
print("To find the raw CSV line number, calculate: Row ID + 2\n")

# =========================================================
# CATEGORY 1: DEMOGRAPHICS, CHRONOLOGY & SEXUAL HISTORY
# =========================================================
print("--- CATEGORY 1: CHRONOLOGY & SEXUAL HISTORY ---\n")

# TEST 1: Absolute Chronological Boundaries
print("TEST 1: Absolute Chronological Boundaries")
bad_fsi = df['First sexual intercourse'] > (df['Age'] + 1e-8)
bad_smokes_years = df['Smokes (years)'] >= (df['Age'] + 1e-8)
bad_hc_years = df['Hormonal Contraceptives (years)'] >= (df['Age'] + 1e-8)
bad_iud_years = df['IUD (years)'] >= (df['Age'] + 1e-8)

chrono_flaws = df[bad_fsi | bad_smokes_years | bad_hc_years | bad_iud_years]
print(f"Found {len(chrono_flaws)} patients violating absolute chronological time limits.")
if len(chrono_flaws) > 0:
    cols = ['Age', 'First sexual intercourse', 'Smokes (years)', 'Hormonal Contraceptives (years)', 'IUD (years)']
    print(chrono_flaws[cols].head(5))
print("-" * 60)

# TEST 2: Clinical Age of Initiation (Biological minimums)
print("TEST 2: Clinical Age of Initiation (Puberty/Biology constraints)")
impossible_hc_start = ((df['Age'] + 1e-8) - df['Hormonal Contraceptives (years)']) < 5
impossible_iud_start = ((df['Age'] + 1e-8) - df['IUD (years)']) < 5
impossible_pregnancies = df['Num of pregnancies'] > ((df['Age'] + 1e-8) - 5) # Roughly >1 preg/year since age 5

clinical_age_flaws = df[impossible_hc_start | impossible_iud_start | impossible_pregnancies]
print(f"Found {len(clinical_age_flaws)} patients violating clinical biological timelines.")
if len(clinical_age_flaws) > 0:
    cols = ['Age', 'Hormonal Contraceptives (years)', 'IUD (years)', 'Num of pregnancies']
    print(clinical_age_flaws[cols].head(5))
print("-" * 60)

# TEST 3: The Virginity / Exposure Paradox
print("TEST 3: The Virginity / Exposure Paradox")
bad_exposure = (df['Number of sexual partners'] == 0) & (
    (df['First sexual intercourse'] > 0) | 
    (df['Num of pregnancies'] > 0) | 
    (df['STDs'] == 1) 
)
exposure_flaws = df[bad_exposure]
print(f"Found {len(exposure_flaws)} patients with 0 partners but positive exposure markers.")
if len(exposure_flaws) > 0:
    cols = ['Number of sexual partners', 'First sexual intercourse', 'Num of pregnancies', 'STDs']
    print(exposure_flaws[cols].head(5))
print("-" * 60 + "\n")


# =========================================================
# CATEGORY 2: LIFESTYLE (SMOKING)
# =========================================================
print("--- CATEGORY 2: LIFESTYLE (SMOKING) ---\n")

# TEST 4: 'Smokes' Master Flag Consistency
print("TEST 4: 'Smokes' Master Flag Consistency")
smokes_flaws = df[
    ((df['Smokes'] == 0) | (df['Smokes'].isnull())) & 
    ((df['Smokes (years)'] > 0) | (df['Smokes (packs/year)'] > 0))
]
print(f"Found {len(smokes_flaws)} patients classified as non-smokers with years/packs > 0.")
if len(smokes_flaws) > 0:
    cols = ['Smokes', 'Smokes (years)', 'Smokes (packs/year)']
    print(smokes_flaws[cols].head(5))
print("-" * 60)

# TEST 5: Pack-Year Math Contradiction
print("TEST 5: Pack-Year Math Contradiction")
pack_year_paradox = (df['Smokes (packs/year)'] > 0) & ((df['Smokes (years)'] == 0) | df['Smokes (years)'].isnull())
smoking_math_flaws = df[pack_year_paradox]
print(f"Found {len(smoking_math_flaws)} patients with pack-years but no documented smoking timeline.")
if len(smoking_math_flaws) > 0:
    cols = ['Smokes (years)', 'Smokes (packs/year)']
    print(smoking_math_flaws[cols].head(5))
print("-" * 60)

# TEST 6: Reverse Logic (0s should mean 0s/NaN)
print("TEST 6: 'Smokes' Reverse Logic")
smokes_reverse_flaws = df[(df['Smokes'] == 0) & ((df['Smokes (years)'].isnull()) | (df['Smokes (packs/year)'].isnull()))]
print(f"Found {len(smokes_reverse_flaws)} non-smokers (0) with NaN instead of 0 in years/packs.")
if len(smokes_reverse_flaws) > 0:
    cols = ['Smokes', 'Smokes (years)', 'Smokes (packs/year)']
    print(smokes_reverse_flaws[cols].head(5))
print("-" * 60 + "\n")


# =========================================================
# CATEGORY 3: CONTRACEPTIVES
# =========================================================
print("--- CATEGORY 3: CONTRACEPTIVES ---\n")

# TEST 7: Hormonal Contraceptives Consistency
print("TEST 7: Hormonal Contraceptives Consistency")
hc_flaws = df[
    ((df['Hormonal Contraceptives'] == 0) | (df['Hormonal Contraceptives'].isnull())) & 
    (df['Hormonal Contraceptives (years)'] > 0)
]
print(f"Found {len(hc_flaws)} patients with contradictory 'Hormonal Contraceptives' data.")
if len(hc_flaws) > 0:
    cols = ['Hormonal Contraceptives', 'Hormonal Contraceptives (years)']
    print(hc_flaws[cols].head(5))
print("-" * 60)

# TEST 8: IUD Consistency
print("TEST 8: IUD Consistency")
iud_flaws = df[
    ((df['IUD'] == 0) | (df['IUD'].isnull())) & 
    (df['IUD (years)'] > 0)
]
print(f"Found {len(iud_flaws)} patients with contradictory 'IUD' data.")
if len(iud_flaws) > 0:
    cols = ['IUD', 'IUD (years)']
    print(iud_flaws[cols].head(5))
print("-" * 60)

# TEST 9: Contraceptives Reverse Logic
print("TEST 9: Contraceptives Reverse Logic (NaNs for Non-users)")
hc_reverse_flaws = df[(df['Hormonal Contraceptives'] == 0) & (df['Hormonal Contraceptives (years)'].isnull())]
iud_reverse_flaws = df[(df['IUD'] == 0) & (df['IUD (years)'].isnull())]
print(f"Found {len(hc_reverse_flaws)} non-HC users and {len(iud_reverse_flaws)} non-IUD users with NaNs instead of 0s.")
if len(hc_reverse_flaws) > 0:
    print("Example HC Reverse Flaws:")
    print(hc_reverse_flaws[['Hormonal Contraceptives', 'Hormonal Contraceptives (years)']].head(3))
if len(iud_reverse_flaws) > 0:
    print("Example IUD Reverse Flaws:")
    print(iud_reverse_flaws[['IUD', 'IUD (years)']].head(3))
print("-" * 60 + "\n")


# =========================================================
# CATEGORY 4: PATHOLOGY & STDs
# =========================================================
print("--- CATEGORY 4: PATHOLOGY & STDs ---\n")

binary_std_cols = [
    'STDs:condylomatosis', 'STDs:cervical condylomatosis', 'STDs:vaginal condylomatosis',
    'STDs:vulvo-perineal condylomatosis', 'STDs:syphilis', 'STDs:pelvic inflammatory disease',
    'STDs:genital herpes', 'STDs:molluscum contagiosum', 'STDs:AIDS', 'STDs:HIV',
    'STDs:Hepatitis B', 'STDs:HPV'
]

# TEST 5: 'STDs' Master Flag Consistency
print("TEST 5: 'STDs' Master Flag Consistency")
stds_flaws = df[
    ((df['STDs'] == 0) | (df['STDs'].isnull())) & 
    (df[binary_std_cols].sum(axis=1) > 0)
]
print(f"Found {len(stds_flaws)} patients with STDs=0 but positive specific disease flags.")
if len(stds_flaws) > 0:
    # Filtering to show only the active disease columns for readability
    active_cols = stds_flaws[binary_std_cols].loc[:, stds_flaws[binary_std_cols].sum() > 0].columns.tolist()
    print(stds_flaws[['STDs'] + active_cols].head(5))
print("-" * 60)

# TEST 11: STD Distinct Count Match
print("TEST 11: STD Distinct Count Match")
count_mismatch = df[binary_std_cols].sum(axis=1) > df['STDs (number)']
std_count_flaws = df[count_mismatch & df['STDs (number)'].notnull()]
print(f"Found {len(std_count_flaws)} patients where binary STD flags exceed the stated 'STDs (number)'.")
if len(std_count_flaws) > 0:
    active_cols = std_count_flaws[binary_std_cols].loc[:, std_count_flaws[binary_std_cols].sum() > 0].columns.tolist()
    print(std_count_flaws[['STDs (number)'] + active_cols].head(5))
print("-" * 60)

# TEST 12: Condylomatosis Hierarchy
print("TEST 12: Condylomatosis Sub-feature Consistency")
condy_cols = ['STDs:cervical condylomatosis', 'STDs:vaginal condylomatosis', 'STDs:vulvo-perineal condylomatosis']
condy_flaws = df[
    ((df['STDs:condylomatosis'] == 0) | (df['STDs:condylomatosis'].isnull())) & 
    (df[condy_cols].sum(axis=1) > 0)
]
print(f"Found {len(condy_flaws)} patients with Master Condylomatosis=0 but localized infections present.")
if len(condy_flaws) > 0:
    print(condy_flaws[['STDs:condylomatosis'] + condy_cols].head(5))
print("-" * 60)

# TEST 13: STD Timeline Contradictions
print("TEST 13: STD Diagnosis Timeline Contradictions")
try:
    col_num_diag = [c for c in df.columns if 'Number of diagnosis' in c][0]
    col_first_diag = [c for c in df.columns if 'first diagnosis' in c][0]
    col_last_diag = [c for c in df.columns if 'last diagnosis' in c][0]
    
    time_paradox = df[col_first_diag] < df[col_last_diag]
    std_timeline_flaws = df[time_paradox]
    print(f"Found {len(std_timeline_flaws)} patients where First Diagnosis occurred AFTER Last Diagnosis.")
    if len(std_timeline_flaws) > 0:
        print(std_timeline_flaws[[col_num_diag, col_first_diag, col_last_diag]].head(5))
except IndexError:
    print("Skipping Timeline Check: Missing timeline columns in DataFrame.")

print("-" * 60 + "\n")
print("PIPELINE COMPLETE.")

PHASE 3: CLINICAL & LOGICAL DATA ASSESSMENT PIPELINE
Note: Row IDs refer to the pandas DataFrame Index.
To find the raw CSV line number, calculate: Row ID + 2

--- CATEGORY 1: CHRONOLOGY & SEXUAL HISTORY ---

TEST 1: Absolute Chronological Boundaries
Found 2 patients violating absolute chronological time limits.
     Age  First sexual intercourse  Smokes (years)  \
312   23                      27.0             0.0   
812   14                      16.0             0.0   

     Hormonal Contraceptives (years)  IUD (years)  
312                             0.00          NaN  
812                             0.08          0.0  
------------------------------------------------------------
TEST 2: Clinical Age of Initiation (Puberty/Biology constraints)
Found 0 patients violating clinical biological timelines.
------------------------------------------------------------
TEST 3: The Virginity / Exposure Paradox
Found 0 patients with 0 partners but positive exposure markers.
-----------------

### Step 1.b: Drop the corrupted rows

In [281]:
# --- Step 1.b: Drop Chronologically Corrupted Rows ---

# Re-create the exact logical mask from TEST 1
bad_fsi = df['First sexual intercourse'] > (df['Age'] + 1e-8)
bad_smokes_years = df['Smokes (years)'] >= (df['Age'] + 1e-8)
bad_hc_years = df['Hormonal Contraceptives (years)'] >= (df['Age'] + 1e-8)
bad_iud_years = df['IUD (years)'] >= (df['Age'] + 1e-8)

chrono_flaws_mask = bad_fsi | bad_smokes_years | bad_hc_years | bad_iud_years

# Drop them dynamically by keeping the inverse of the mask
patients_before = len(df)
df = df[~chrono_flaws_mask].reset_index(drop=True)
patients_after = len(df)

print(f"Dropped {patients_before - patients_after} patients due to absolute chronological impossibilities.")
print(f"New dataset shape: {df.shape}")

Dropped 2 patients due to absolute chronological impossibilities.
New dataset shape: (856, 36)


### Step 2: Missing data handling

In [282]:
print("--- Step 2: Missing Data Assessment ---")

# Calculate the percentage of missing values per column
# df.isnull().sum() counts the NaNs, dividing by len(df) gets the ratio, * 100 gets percentage
missing_percentages = (df.isnull().sum() / len(df)) * 100

# Filter to only show columns that have AT LEAST SOME missing data (> 0%)
missing_cols = missing_percentages[missing_percentages > 0].sort_values(ascending=False)

print(f"Total columns with missing data: {len(missing_cols)} out of {df.shape[1]}\n")

if len(missing_cols) > 0:
    print("--- SEVERE Missing Data (> 50%) [Drop Candidates] ---")
    severe_cols = missing_cols[missing_cols > 50]
    if len(severe_cols) > 0:
        print(severe_cols.round(2).astype(str) + ' %')
    else:
        print("None")
        
    print("\n--- MODERATE/MINOR Missing Data (<= 50%) [Imputation Candidates] ---")
    moderate_cols = missing_cols[missing_cols <= 50]
    if len(moderate_cols) > 0:
        print(moderate_cols.round(2).astype(str) + ' %')
    else:
        print("None")
print("\n" + "="*40 + "\n")

--- Step 2: Missing Data Assessment ---
Total columns with missing data: 26 out of 36

--- SEVERE Missing Data (> 50%) [Drop Candidates] ---
STDs: Time since first diagnosis    91.71 %
STDs: Time since last diagnosis     91.71 %
dtype: object

--- MODERATE/MINOR Missing Data (<= 50%) [Imputation Candidates] ---
IUD (years)                           13.55 %
IUD                                   13.55 %
Hormonal Contraceptives               12.62 %
Hormonal Contraceptives (years)       12.62 %
STDs (number)                         12.27 %
STDs                                  12.27 %
STDs:vulvo-perineal condylomatosis    12.27 %
STDs:vaginal condylomatosis           12.27 %
STDs:pelvic inflammatory disease      12.27 %
STDs:syphilis                         12.27 %
STDs:genital herpes                   12.27 %
STDs:molluscum contagiosum            12.27 %
STDs:HIV                              12.27 %
STDs:AIDS                             12.27 %
STDs:condylomatosis                   12.27

# Phase 4: Handling Missing Data (Imputation)

In [283]:
print("==========================================")
print("PHASE 4: HANDLING MISSING DATA (IMPUTATION)")
print("==========================================")

# ---------------------------------------------------------
# 1. Drop the "Not Applicable" Columns
# ---------------------------------------------------------
cols_to_drop = ['STDs: Time since first diagnosis', 'STDs: Time since last diagnosis']
df = df.drop(columns=cols_to_drop, errors='ignore')
print(f"Dropped {len(cols_to_drop)} columns due to >90% missing data: {cols_to_drop}\n")

# ---------------------------------------------------------
# 2. Capture Missingness as a Feature (Shadow Matrix)
# ---------------------------------------------------------
print("--- Generating Missingness Indicators ---")
missing_cols = df.columns[df.isnull().any()].tolist()
indicator_count = 0
added_indicators = []

for col in missing_cols:
    # Only flag columns with > 5% missingness to avoid noise
    if (df[col].isnull().sum() / len(df)) > 0.05:
        indicator_name = f"{col}_is_missing"
        df[indicator_name] = df[col].isnull().astype(int)
        added_indicators.append(indicator_name)
        indicator_count += 1

print(f"Created {indicator_count} new binary features to capture clinical missingness patterns:")
print(added_indicators)
print("\n")

PHASE 4: HANDLING MISSING DATA (IMPUTATION)
Dropped 2 columns due to >90% missing data: ['STDs: Time since first diagnosis', 'STDs: Time since last diagnosis']

--- Generating Missingness Indicators ---
Created 19 new binary features to capture clinical missingness patterns:
['Num of pregnancies_is_missing', 'Hormonal Contraceptives_is_missing', 'Hormonal Contraceptives (years)_is_missing', 'IUD_is_missing', 'IUD (years)_is_missing', 'STDs_is_missing', 'STDs (number)_is_missing', 'STDs:condylomatosis_is_missing', 'STDs:cervical condylomatosis_is_missing', 'STDs:vaginal condylomatosis_is_missing', 'STDs:vulvo-perineal condylomatosis_is_missing', 'STDs:syphilis_is_missing', 'STDs:pelvic inflammatory disease_is_missing', 'STDs:genital herpes_is_missing', 'STDs:molluscum contagiosum_is_missing', 'STDs:AIDS_is_missing', 'STDs:HIV_is_missing', 'STDs:Hepatitis B_is_missing', 'STDs:HPV_is_missing']




### Option A: Imputing with median

In [284]:
# # =========================================================
# # OPTION A: MEDIAN IMPUTATION
# # =========================================================
# print("--- Deterministic Conditional Imputation (Median) ---")
# conditional_groups = [
#     ('IUD', ['IUD (years)']), 
#     ('Hormonal Contraceptives', ['Hormonal Contraceptives (years)']),
#     ('Smokes', ['Smokes (years)', 'Smokes (packs/year)'])
# ]

# for master, sub_cols in conditional_groups:
#     master_missing = df[master].isnull().sum()
#     master_median = df[master].median()
#     df[master] = df[master].fillna(master_median)
#     print(f"-> '{master}': Filled {master_missing} missing values with median {master_median}.")
    
#     for sub in sub_cols:
#         sub_missing = df[sub].isnull().sum()
#         user_median = df[df[master] == 1][sub].median()
        
#         # Track counts for logging
#         zero_fills = (df[master] == 0) & df[sub].isnull()
#         median_fills = (df[master] == 1) & df[sub].isnull()
        
#         df.loc[df[master] == 0, sub] = df.loc[df[master] == 0, sub].fillna(0)
#         df.loc[df[master] == 1, sub] = df.loc[df[master] == 1, sub].fillna(user_median)
        
#         print(f"   - '{sub}': Filled {sub_missing} total missing values.")
#         print(f"     (Filled {zero_fills.sum()} with 0.0; Filled {median_fills.sum()} with user median {user_median:.2f})")

# print("\n--- Global Fallback Imputation (Median) ---")
# fallback_count = 0
# for col in df.columns:
#     missing_count = df[col].isnull().sum()
#     if missing_count > 0:
#         col_median = df[col].median()
#         df[col] = df[col].fillna(col_median)
#         print(f"-> '{col}': Filled {missing_count} missing values with global median {col_median}.")
#         fallback_count += 1

# if fallback_count == 0:
#     print("No remaining columns required fallback imputation.")
# print("\nMedian Imputation Complete.\n")

### Option B: Imputing with KNN

In [ ]:
# =========================================================
# OPTION B: K-NEAREST NEIGHBORS (KNN) IMPUTATION
# =========================================================
print("--- Deterministic Pre-Fill ---")
conditional_groups = [('IUD', ['IUD (years)']), ('Hormonal Contraceptives', ['Hormonal Contraceptives (years)']), ('Smokes', ['Smokes (years)', 'Smokes (packs/year)'])]
for master, sub_cols in conditional_groups:
    df[master] = df[master].fillna(df[master].median())
    for sub in sub_cols:
        fills = ((df[master] == 0) & df[sub].isnull()).sum()
        df.loc[df[master] == 0, sub] = df.loc[df[master] == 0, sub].fillna(0)
        print(f"-> '{sub}': Pre-filled {fills} NaNs with 0.0 because master '{master}' == 0.")

print("\n--- KNN Imputation Execution ---")
binary_cols = [col for col in df.columns if set(df[col].dropna().unique()).issubset({0.0, 1.0})]
missing_mask = df.isnull()

scaler = MinMaxScaler()
df_scaled = pd.DataFrame(scaler.fit_transform(df), columns=df.columns, index=df.index)

imputer = KNNImputer(n_neighbors=5, weights='distance')
df_imputed_scaled = pd.DataFrame(imputer.fit_transform(df_scaled), columns=df.columns, index=df.index)
df = pd.DataFrame(scaler.inverse_transform(df_imputed_scaled), columns=df.columns, index=df.index)

print("   [Audit: Raw KNN Imputed Values Before Rounding]")
knn_imputed_count = 0
for col in df.columns:
    if missing_mask[col].any():
        imputed_raw_vals = df.loc[missing_mask[col], col]
        missing_count = missing_mask[col].sum()
        print(f"   -> '{col}' ({missing_count} missing) | Min: {imputed_raw_vals.min():.4f}, Max: {imputed_raw_vals.max():.4f}, Mean: {imputed_raw_vals.mean():.4f}")
        knn_imputed_count += missing_count

# --- Domain-Aware Post-KNN Rounding ---
print("--- Enforcing Clinical Data Domains ---")

# 1. Binary Domain
for col in binary_cols:
    if missing_mask[col].any():
        df[col] = np.round(df[col])

# 2. Discrete Count Domain
discrete_cols = ['Number of sexual partners', 'Num of pregnancies', 'STDs (number)', 'STDs: Number of diagnosis']
for col in discrete_cols:
    if col in df.columns and missing_mask[col].any():
        df[col] = np.round(df[col])
        print(f"-> Forced '{col}' back to strict integers.")

# 3. Continuous Domain (Years/Packs)
# We intentionally do NOT round these. KNN's float estimations (e.g., 2.3 years) remain intact.
print("-> Maintained continuous floats for time/volume variables (Years, Packs).")
        
print(f"\nTotal values dynamically guessed via KNN: {knn_imputed_count}")
print("KNN Imputation Complete.\n")

--- Deterministic Pre-Fill ---
-> 'IUD (years)': Pre-filled 116 NaNs with 0.0 because master 'IUD' == 0.
-> 'Hormonal Contraceptives (years)': Pre-filled 0 NaNs with 0.0 because master 'Hormonal Contraceptives' == 0.
-> 'Smokes (years)': Pre-filled 13 NaNs with 0.0 because master 'Smokes' == 0.
-> 'Smokes (packs/year)': Pre-filled 13 NaNs with 0.0 because master 'Smokes' == 0.

--- KNN Imputation Execution ---


   [Audit: Raw KNN Imputed Values Before Rounding]
   -> 'Number of sexual partners' (26 missing) | Min: 1.0000, Max: 3.6209, Mean: 2.1461
   -> 'First sexual intercourse' (7 missing) | Min: 16.0000, Max: 20.8685, Mean: 18.4266
   -> 'Num of pregnancies' (55 missing) | Min: 1.0000, Max: 4.9332, Mean: 2.2579
   -> 'Hormonal Contraceptives (years)' (108 missing) | Min: 0.7980, Max: 9.7826, Mean: 7.0691
   -> 'STDs' (105 missing) | Min: 0.0000, Max: 0.4020, Mean: 0.3430
   -> 'STDs (number)' (105 missing) | Min: 0.0000, Max: 0.5994, Mean: 0.4649
   -> 'STDs:condylomatosis' (105 missing) | Min: 0.0000, Max: 0.2101, Mean: 0.1219
   -> 'STDs:cervical condylomatosis' (105 missing) | Min: 0.0000, Max: 0.0000, Mean: 0.0000
   -> 'STDs:vaginal condylomatosis' (105 missing) | Min: 0.0000, Max: 0.0000, Mean: 0.0000
   -> 'STDs:vulvo-perineal condylomatosis' (105 missing) | Min: 0.0000, Max: 0.2101, Mean: 0.1219
   -> 'STDs:syphilis' (105 missing) | Min: 0.0000, Max: 0.4004, Mean: 0.0703
   -> 'STD

In [286]:
# ---------------------------------------------------------
# Post-Imputation Clinical Override Firewall (Strict Audit Trail)
# ---------------------------------------------------------
print("--- Post-Imputation Consistency Check & Override ---")
corrections_made = 0

# 1. Absolute Chronological Ceilings
fsi_conflict = df['First sexual intercourse'] > (df['Age'] + 1e-8)
if fsi_conflict.sum() > 0:
    print(f"-> OVERRIDE: Capped 'First sexual intercourse' at Age for {fsi_conflict.sum()} rows.")
    audit_df = df.loc[fsi_conflict, ['Age', 'First sexual intercourse']].copy()
    audit_df.rename(columns={'First sexual intercourse': 'FSI (Imputed)'}, inplace=True)
    
    df.loc[fsi_conflict, 'First sexual intercourse'] = df.loc[fsi_conflict, 'Age']
    audit_df['FSI (Corrected)'] = df.loc[fsi_conflict, 'First sexual intercourse']
    
    print(audit_df.head(5).to_string())
    print("")
    corrections_made += fsi_conflict.sum()

duration_cols = ['Smokes (years)', 'Hormonal Contraceptives (years)', 'IUD (years)']
for col in duration_cols:
    if col in df.columns:
        dur_conflict = df[col] > (df['Age'] + 1e-8)
        if dur_conflict.sum() > 0:
            print(f"-> OVERRIDE: Capped '{col}' at Age for {dur_conflict.sum()} rows.")
            audit_df = df.loc[dur_conflict, ['Age', col]].copy()
            audit_df.rename(columns={col: f'{col} (Imputed)'}, inplace=True)
            
            df.loc[dur_conflict, col] = df.loc[dur_conflict, 'Age']
            audit_df[f'{col} (Corrected)'] = df.loc[dur_conflict, col]
            
            print(audit_df.head(5).to_string())
            print("")
            corrections_made += dur_conflict.sum()

# 2. Clinical Age of Initiation (Puberty constraints: ~Age 5)
for col in duration_cols:
    if col in df.columns:
        bio_conflict = ((df['Age'] + 1e-8) - df[col]) < 5
        if bio_conflict.sum() > 0:
            print(f"-> OVERRIDE: Curtailed '{col}' to biologically possible timeframe for {bio_conflict.sum()} rows.")
            audit_df = df.loc[bio_conflict, ['Age', col]].copy()
            audit_df.rename(columns={col: f'{col} (Imputed)'}, inplace=True)
            
            df.loc[bio_conflict, col] = df.loc[bio_conflict, 'Age'] - 5
            audit_df[f'{col} (Corrected)'] = df.loc[bio_conflict, col]
            
            print(audit_df.head(5).to_string())
            print("")
            corrections_made += bio_conflict.sum()

years_active = (df['Age'] + 1e-8) - df['First sexual intercourse']
preg_conflict = df['Num of pregnancies'] > years_active
if preg_conflict.sum() > 0:
    print(f"-> OVERRIDE: Capped 'Num of pregnancies' at max possible active years for {preg_conflict.sum()} rows.")
    audit_df = df.loc[preg_conflict, ['Age', 'First sexual intercourse', 'Num of pregnancies']].copy()
    audit_df.rename(columns={'Num of pregnancies': 'Pregnancies (Imputed)'}, inplace=True)
    audit_df['Active Years Limit'] = np.floor(years_active[preg_conflict])
    
    df.loc[preg_conflict, 'Num of pregnancies'] = np.floor(years_active[preg_conflict])
    audit_df['Pregnancies (Corrected)'] = df.loc[preg_conflict, 'Num of pregnancies']
    
    print(audit_df.head(5).to_string())
    print("")
    corrections_made += preg_conflict.sum()

# 3. Virginity / Exposure Paradox
exposure_conflict = (df['Number of sexual partners'] == 0) & ((df['First sexual intercourse'] > 0) | (df['Num of pregnancies'] > 0))
if exposure_conflict.sum() > 0:
    print(f"-> OVERRIDE: Forced 'Number of sexual partners' to 1 for {exposure_conflict.sum()} rows due to exposure paradox.")
    audit_df = df.loc[exposure_conflict, ['First sexual intercourse', 'Num of pregnancies', 'Number of sexual partners']].copy()
    audit_df.rename(columns={'Number of sexual partners': 'Partners (Imputed)'}, inplace=True)
    
    df.loc[exposure_conflict, 'Number of sexual partners'] = 1
    audit_df['Partners (Corrected)'] = df.loc[exposure_conflict, 'Number of sexual partners']
    
    print(audit_df.head(5).to_string())
    print("")
    corrections_made += exposure_conflict.sum()

# 4. Smoking Math Paradox
smoke_math_conflict = (df['Smokes (packs/year)'] > 0) & (df['Smokes (years)'] == 0)
if smoke_math_conflict.sum() > 0:
    print(f"-> OVERRIDE: Forced 'Smokes (years)' to 1 for {smoke_math_conflict.sum()} rows due to positive pack/years.")
    audit_df = df.loc[smoke_math_conflict, ['Smokes (packs/year)', 'Smokes (years)']].copy()
    audit_df.rename(columns={'Smokes (years)': 'Smokes Yrs (Imputed)'}, inplace=True)
    
    df.loc[smoke_math_conflict, 'Smokes (years)'] = 1
    audit_df['Smokes Yrs (Corrected)'] = df.loc[smoke_math_conflict, 'Smokes (years)']
    
    print(audit_df.head(5).to_string())
    print("")
    corrections_made += smoke_math_conflict.sum()

# 5. Master vs Sub-Feature Logic (STDs)
all_std_related_cols = [
    col for col in df.columns 
    if col.startswith('STDs') 
    and col not in ['STDs', 'STDs (number)'] 
    and not col.endswith('_is_missing')
]

std_conflict = (df['STDs'] == 0) & (df[all_std_related_cols].sum(axis=1) > 0)
if std_conflict.sum() > 0:
    print(f"-> OVERRIDE: Forced master 'STDs' to 1.0 for {std_conflict.sum()} rows (Sub-STDs detected).")
    active_cols = [c for c in all_std_related_cols if df.loc[std_conflict, c].sum() > 0][:3]
    audit_df = df.loc[std_conflict, ['STDs'] + active_cols].copy()
    audit_df.rename(columns={'STDs': 'STDs Master (Imputed)'}, inplace=True)
    
    df.loc[std_conflict, 'STDs'] = 1.0
    audit_df['STDs Master (Corrected)'] = df.loc[std_conflict, 'STDs']
    
    print(audit_df.head(5).to_string())
    print("")
    corrections_made += std_conflict.sum()

# 6. Condylomatosis Hierarchy
condy_cols = [
    col for col in df.columns 
    if 'condylomatosis' in col 
    and col != 'STDs:condylomatosis' 
    and not col.endswith('_is_missing')
]

condy_conflict = (df['STDs:condylomatosis'] == 0) & (df[condy_cols].sum(axis=1) > 0)
if condy_conflict.sum() > 0:
    print(f"-> OVERRIDE: Forced 'STDs:condylomatosis' to 1.0 for {condy_conflict.sum()} rows.")
    audit_df = df.loc[condy_conflict, ['STDs:condylomatosis'] + condy_cols].copy()
    audit_df.rename(columns={'STDs:condylomatosis': 'Condy Master (Imputed)'}, inplace=True)
    
    df.loc[condy_conflict, 'STDs:condylomatosis'] = 1.0
    audit_df['Condy Master (Corrected)'] = df.loc[condy_conflict, 'STDs:condylomatosis']
    
    print(audit_df.head(5).to_string())
    print("")
    corrections_made += condy_conflict.sum()

print(f"Total programmatic boundary corrections executed on imputed data: {corrections_made}")

# Final Proof
print("\n--- Final Dataset Check ---")
print(f"Total missing values left in the ENTIRE dataset: {df.isnull().sum().sum()}")
print(f"Final Dataset Shape: {df.shape[0]} patients, {df.shape[1]} features")
print("==========================================\n")

--- Post-Imputation Consistency Check & Override ---
-> OVERRIDE: Capped 'Num of pregnancies' at max possible active years for 33 rows.
      Age  First sexual intercourse  Pregnancies (Imputed)  Active Years Limit  Pregnancies (Corrected)
7    26.0                      26.0                    3.0                 0.0                      0.0
89   33.0                      32.0                    2.0                 1.0                      1.0
113  23.0                      23.0                    2.0                 0.0                      0.0
128  29.0                      29.0                    1.0                 0.0                      0.0
169  18.0                      18.0                    4.0                 0.0                      0.0

Total programmatic boundary corrections executed on imputed data: 33

--- Final Dataset Check ---
Total missing values left in the ENTIRE dataset: 0
Final Dataset Shape: 856 patients, 53 features



# Phase 5: Export clean data

In [287]:
# Phase 5: Export Clean Data
# Save the cleaned dataset so we can pick it up in the next notebook
output_path = "../data/clean_cervical_cancer_data_ntb.csv"
df.to_csv(output_path, index=False)

print(f"Clean data successfully saved to {output_path}")

Clean data successfully saved to ../data/clean_cervical_cancer_data_ntb.csv
